# Phase 6: Analytical Approach - Analysis v2

This notebook implements the statistical analyses described in the manuscript § 3.5.

## Analyses Included

1. **MCQ Position Bias Analysis (§ 3.5.1)**
   - Visual diagnostics: accuracy by position, deviation from uniform
   - Chi-square test of independence
   - Friedman test (repeated measures)
   - Effect size: Cramér's V

2. **OSQ-Specific Analysis (§ 3.5.2)**
   - Score distributions per model
   - Judge-level distributions
   - Rubric dimension analysis
   - Inter-judge agreement (Spearman correlation)
   - Bootstrap confidence intervals

3. **Format Comparison: MCQ vs OSQ (§ 3.5.3)**
   - MCQ-OSQ scatter plots (multi-judge)
   - Correlation analysis (Pearson, Spearman)
   - Wilcoxon signed-rank test
   - Cohen's d effect size

4. **Tokenomics & Cost-Effectiveness (§ 3.5.4)**
   - Token and cost metrics
   - Cost-quality trade-off visualizations
   - Efficiency analysis

## Dataset

- **MCQ**: 1144 questions total (position variants a/b/c/d)
- **OSQ**: 845 questions (matched subset with MCQ)
- **Matched**: 845 questions for MCQ vs OSQ comparison

---
## 1. SETUP & DATA LOADING

### 1.1 Imports

In [1]:
# Standard libraries
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats
from scipy.stats import (
    chi2_contingency,          # Chi-square test
    friedmanchisquare,         # Friedman test
    pearsonr, spearmanr,       # Correlations
    wilcoxon,                  # Paired difference test
    bootstrap,                 # Bootstrap CI
    linregress,                # Linear regression
    gaussian_kde               # KDE for distributions
)

# Progress
from tqdm.auto import tqdm
from itertools import combinations

# Token counting
import tiktoken

# Local parsers
import sys
from parsers import (
    parse_mcq_samples, 
    parse_osq_judged_samples, 
    align_mcq_osq_results,
    filter_osq_data, 
    get_available_judges_and_prompts
)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✅ Imports complete')

✅ Imports complete


### 1.2 Configuration

In [2]:
# Paths
phase4_dir = Path('../phase4_inference/output')
phase5_dir = Path('../phase5_llm_as_a_judge')
output_dir = Path('output_v2')
output_dir.mkdir(exist_ok=True)

# Constants
N_MCQ_QUESTIONS = 1144  # Total MCQ questions
N_MATCHED_QUESTIONS = 845  # Matched with OSQ
N_BOOTSTRAP = 1000  # Bootstrap iterations
CONFIDENCE_LEVEL = 0.95  # 95% CI

# Position mapping: variants to positions
POSITION_MAPPING = {
    'a': 'A',
    'b': 'B',
    'c': 'C',
    'd': 'D'
}

# Judge selection configuration
# Options: 'all' or list of specific judge names
JUDGE_SELECTION = 'all'

print(f'✅ Configuration complete')
print(f'   Output directory: {output_dir.absolute()}')
print(f'   MCQ questions: {N_MCQ_QUESTIONS}')
print(f'   Matched questions: {N_MATCHED_QUESTIONS}')
print(f'   Bootstrap iterations: {N_BOOTSTRAP}')

✅ Configuration complete
   Output directory: c:\Users\rabel\Desktop\dissertation\src\phase6_analysis\output_v2
   MCQ questions: 1144
   Matched questions: 845
   Bootstrap iterations: 1000


### 1.3 Helper Functions

In [3]:
def export_latex_table(df, filename, caption, label, column_format=None):
    """
    Export DataFrame to LaTeX table with booktabs formatting.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Data to export
    filename : str
        Output filename (without path)
    caption : str
        Table caption
    label : str
        LaTeX label for referencing
    column_format : str, optional
        LaTeX column format (e.g., 'lrrr')
    """
    if column_format is None:
        # Auto-generate: 'l' for first column, 'r' for others
        column_format = 'l' + 'r' * (len(df.columns) - 1)
    
    latex_str = df.to_latex(
        index=True,
        column_format=column_format,
        escape=False,
        float_format='%.4f',
        caption=caption,
        label=label
    )
    
    output_path = output_dir / filename
    with open(output_path, 'w') as f:
        f.write(latex_str)
    
    print(f'   ✓ Exported: {filename}')


def identify_complete_judges(osq_df, n_required_questions=845):
    """
    Identify judges that have complete data (all questions for all models).
    A judge is complete if every model has attempted the required number of questions
    (i.e., provided a response or was properly scored, excluding judge/API errors).
    
    Parameters:
    -----------
    osq_df : pd.DataFrame
        OSQ judged samples with parse_status field
    n_required_questions : int
        Number of questions required per model
    
    Returns:
    --------
    list : Judge names with complete data
    """
    complete_judges = []
    
    for judge in osq_df['judge_model'].unique():
        judge_df = osq_df[osq_df['judge_model'] == judge]
        
        # Count samples where model attempted (success or model_error, excluding judge/API errors)
        valid_attempts = judge_df[judge_df['parse_status'].isin(['success', 'model_error'])]
        questions_per_model = valid_attempts.groupby('model')['question_id'].nunique()
        
        if (questions_per_model >= n_required_questions).all():
            complete_judges.append(judge)
    
    return complete_judges


def filter_by_judge_selection(osq_df, judge_selection='all'):
    """
    Filter OSQ data by judge selection.
    
    Parameters:
    -----------
    osq_df : pd.DataFrame
        OSQ judged samples
    judge_selection : str or list
        'all' for all judges, or list of judge names
    
    Returns:
    --------
    pd.DataFrame : Filtered OSQ data
    """
    if judge_selection == 'all':
        return osq_df
    else:
        return osq_df[osq_df['judge_model'].isin(judge_selection)]


print('✅ Helper functions defined')

✅ Helper functions defined


### 1.4 Load Data

#### Load MCQ as mcq_df

In [4]:
print('📂 Loading MCQ samples...')

mcq_df = parse_mcq_samples(phase4_dir, use_latest=True)
mcq_df = pd.DataFrame(mcq_df)
# mcq_df = parse_mcq_samples(phase4_dir)
print(f'   Loaded {len(mcq_df)} MCQ samples')
print(f'   Models: {mcq_df["model"].nunique()}')
print(f'   Questions: {mcq_df["question_id"].nunique()}')
print(f'   Variants: {sorted(mcq_df["variant"].unique())}')

📂 Loading MCQ samples...
📖 Parsing sysengbench/anthropic__claude-sonnet-4.5/samples_sysengbench_2025-11-16T04-31-27.456510.jsonl
📖 Parsing sysengbench/devstral__24b/samples_sysengbench_2025-11-19T22-11-13.670981.jsonl
📖 Parsing sysengbench/gemma3__12b/samples_sysengbench_2025-11-19T22-03-16.047055.jsonl
📖 Parsing sysengbench/gemma3__1b/samples_sysengbench_2025-11-21T00-02-09.310319.jsonl
📖 Parsing sysengbench/gemma3__27b/samples_sysengbench_2025-11-14T23-11-53.059743.jsonl
📖 Parsing sysengbench/gemma3__4b/samples_sysengbench_2025-11-21T00-19-55.392185.jsonl
📖 Parsing sysengbench/google__gemini-2.5-flash/samples_sysengbench_2025-11-16T03-13-38.857265.jsonl
📖 Parsing sysengbench/llama3.2__1b/samples_sysengbench_2025-11-21T01-33-27.866061.jsonl
📖 Parsing sysengbench/llama3.2__3b/samples_sysengbench_2025-11-19T21-42-16.985970.jsonl
📖 Parsing sysengbench/llama3.3__70b/samples_sysengbench_2025-11-15T01-10-38.229116.jsonl
📖 Parsing sysengbench/llama4__16x17b/samples_sysengbench_2025-11-15T01-

#### Load OSQ as osq_df

In [5]:
print('\n📂 Loading OSQ judged samples...')
osq_df = parse_osq_judged_samples(phase5_dir)
osq_df = pd.DataFrame(osq_df)
print(f'   Loaded {len(osq_df)} OSQ samples')
print(f'   Models: {osq_df["model"].nunique()}')
print(f'   Questions: {osq_df["question_id"].nunique()}')
print(f'   Judges: {sorted(osq_df["judge_model"].unique())}')


📂 Loading OSQ judged samples...
📖 Parsing anthropic__claude-sonnet-4.5/samples_sysengbench-osq_2025-11-16T21-02-49.453852__gpt-oss_120b-p1.jsonl (judge: gpt-oss_120b, prompt: p1)
   ⚠️  Line 107: judge error - missing scores ['ta', 'cu', 'co', 'cl', 'pr']
   ⚠️  Line 110: judge error - missing scores ['ta', 'cu', 'co', 'cl', 'pr']
   ⚠️  Line 111: judge error - missing scores ['ta', 'cu', 'co', 'cl', 'pr']
   ⚠️  Line 748: API error - missing_fields: ['student_response']
📖 Parsing anthropic__claude-sonnet-4.5/samples_sysengbench-osq_2025-11-16T21-02-49.453852__openai_gpt-5-mini-p1.jsonl (judge: openai_gpt-5-mini, prompt: p1)
   ⚠️  Line 748: API error - missing_fields: ['student_response']
📖 Parsing anthropic__claude-sonnet-4.5/samples_sysengbench-osq_2025-11-16T21-02-49.453852__openai_gpt-5-p1.jsonl (judge: openai_gpt-5, prompt: p1)
   ⚠️  Line 298: API error - Response ended prematurely
   ⚠️  Line 510: judge error - missing scores ['ta', 'cu', 'co', 'cl', 'pr']
   ⚠️  Line 612: API

#### Checking the OSQ judging progress matrix 

In [6]:
def build_osq_multi_judge_progress_matrix(phase5_dir: Path, use_latest: bool = False) -> pd.DataFrame:
    """
    Build progress matrix showing ALL judge models × ALL evaluated models.
    
    Args:
        phase5_dir: Path to phase5_llm_as_a_judge directory
        use_latest: If True, only show latest judge per model; if False, show all judges
    
    Returns:
        DataFrame with columns: model_name, judge_model, prompt_id, sample_count, judged_file
    """
    import re
    from pathlib import Path
    
    osq_judge_dir = phase5_dir / "sysengbench-osq-llm-judge"
    
    if not osq_judge_dir.exists():
        print(f"⚠️  OSQ judge directory not found: {osq_judge_dir}")
        return pd.DataFrame()
    
    rows = []
    
    # Iterate through each model directory
    model_dirs = sorted([d for d in osq_judge_dir.iterdir() if d.is_dir()])
    
    for model_dir in model_dirs:
        model_name = model_dir.name
        
        # Find all judge files
        judge_files = sorted([f for f in model_dir.iterdir() 
                            if f.is_file() and f.name.startswith('samples_') and f.suffix == '.jsonl'])
        
        if not judge_files:
            continue
        
        # Group by judge model and prompt
        judge_data = []
        for jf in judge_files:
            # Extract judge model and prompt ID from filename
            # Format: samples_sysengbench-osq_TIMESTAMP__JUDGE_MODEL-PROMPT.jsonl
            match = re.search(r'__(.+)-(p\d+)\.jsonl$', jf.name)
            if match:
                judge_model = match.group(1)
                prompt_id = match.group(2)
                
                # Count samples
                try:
                    with open(jf, 'r', encoding='utf-8') as f:
                        sample_count = sum(1 for _ in f)
                except:
                    sample_count = 0
                
                judge_data.append({
                    'file': jf,
                    'judge_model': judge_model,
                    'prompt_id': prompt_id,
                    'sample_count': sample_count,
                    'timestamp': jf.stat().st_mtime
                })
        
        # Filter to latest if requested
        if use_latest and judge_data:
            # Sort by timestamp and take the latest
            judge_data = sorted(judge_data, key=lambda x: x['timestamp'], reverse=True)
            judge_data = [judge_data[0]]
        
        # Add to rows
        for jd in judge_data:
            rows.append({
                'model_name': model_name,
                'judge_model': jd['judge_model'],
                'prompt_id': jd['prompt_id'],
                'sample_count': jd['sample_count'],
                'judged_file': str(jd['file'].name)
            })
    
    df = pd.DataFrame(rows)
    
    if len(df) > 0:
        # Sort for better readability
        df = df.sort_values(['judge_model', 'model_name']).reset_index(drop=True)
    
    return df

print("✓ Multi-judge progress matrix function loaded")

✓ Multi-judge progress matrix function loaded


In [7]:
# Configuration: Judge Loading Strategy
# Set to False to load ALL available judges (recommended for comprehensive analysis)
# Set to True to load only the latest judge per model
OSQ_USE_LATEST = False  # Load ALL judges to see complete judgment matrix

print(f"Judge Loading Mode: {'LATEST ONLY' if OSQ_USE_LATEST else 'ALL JUDGES'}")

Judge Loading Mode: ALL JUDGES


In [8]:
# Build and display OSQ multi-judge progress matrix
print("=" * 80)
print("OSQ MULTI-JUDGE PROGRESS MATRIX")
print("=" * 80)

osq_progress_df = build_osq_multi_judge_progress_matrix(phase5_dir, use_latest=OSQ_USE_LATEST)

if len(osq_progress_df) > 0:
    print(f"\n📊 Found {len(osq_progress_df)} judge-model combinations")
    print(f"   Judges: {osq_progress_df['judge_model'].nunique()}")
    print(f"   Models: {osq_progress_df['model_name'].nunique()}")
    print(f"   Prompts: {osq_progress_df['prompt_id'].unique().tolist()}")
    
    # Display summary by judge
    print("\n📋 Judge Model Summary:")
    judge_summary = osq_progress_df.groupby('judge_model').agg({
        'model_name': 'count',
        'sample_count': 'sum'
    }).rename(columns={'model_name': 'models_judged', 'sample_count': 'total_samples'})
    display(judge_summary)
    
    # Create pivot table for matrix view
    print("\n📊 Progress Matrix (Judge × Model):")
    matrix_pivot = osq_progress_df.pivot_table(
        index='judge_model',
        columns='model_name',
        values='sample_count',
        fill_value=0
    ).astype(int)
    display(matrix_pivot)
    
    # Full detail table
    print("\n📋 Complete Judge-Model Combinations:")
    display(osq_progress_df)
    
else:
    print("⚠️  No OSQ judge data found")

print("\n" + "=" * 80)

OSQ MULTI-JUDGE PROGRESS MATRIX

📊 Found 51 judge-model combinations
   Judges: 3
   Models: 19
   Prompts: ['p1']

📋 Judge Model Summary:


,models_judged,total_samples
judge_model,,
gpt-oss_120b,19,16055
openai_gpt-5,13,8850
openai_gpt-5-mini,19,16055



📊 Progress Matrix (Judge × Model):


model_name,anthropic__claude-sonnet-4.5,devstral__24b,gemma3__12b,gemma3__1b,gemma3__27b,gemma3__4b,google__gemini-2.5-flash,llama3.2__1b,llama3.2__3b,llama3.3__70b,llama4__16x17b,mistral-large__123b,mistral-small3.2__24b,mixtral__8x22b,openai__gpt-4.1,phi3.5__3.8b,phi3__14b,phi4-mini__3.8b,phi4__14b
judge_model,,,,,,,,,,,,,,,,,,,
gpt-oss_120b,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845
openai_gpt-5,845,845,845,845,845,845,845,845,845,845,0,0,0,0,200,0,0,0,200
openai_gpt-5-mini,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845,845



📋 Complete Judge-Model Combinations:


,model_name,judge_model,prompt_id,sample_count,judged_file
0,anthropic__claude-sonnet-4.5,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-16T21-02-49.45...
1,devstral__24b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-19T22-36-33.28...
2,gemma3__12b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-19T22-23-32.10...
3,gemma3__1b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-21T00-11-57.48...
4,gemma3__27b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-15T01-24-12.25...
5,gemma3__4b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-21T00-27-28.70...
6,google__gemini-2.5-flash,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-16T04-10-37.21...
7,llama3.2__1b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-20T23-59-07.91...
8,llama3.2__3b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-19T21-47-50.05...
9,llama3.3__70b,gpt-oss_120b,p1,845,samples_sysengbench-osq_2025-11-14T23-24-28.91...


#### Filter OSQ to desired judges as osq_filtered

In [9]:
# Optional: Filter OSQ data by specific judge(s) and/or prompt
# FILTER_JUDGE can be a single string or a list of judge names

# Examples:
# FILTER_JUDGE = None                                  # All judges pass through
# FILTER_JUDGE = 'openai_gpt-5'                        # Single judge
# FILTER_JUDGE = ['openai_gpt-5-mini', 'openai_gpt-5'] # Multiple judges
# FILTER_PROMPT = 'p1'                                 # Single prompt

# FILTER_JUDGE = None  # Set to judge name(s) or None for all
FILTER_JUDGE = ['openai_gpt-5-mini', 'gpt-oss_120b'] # Multiple judges
FILTER_PROMPT = None  # Set to prompt ID or None for all

if 'osq_df' in locals() and (FILTER_JUDGE is not None or FILTER_PROMPT is not None):
    print("\n" + "=" * 80)
    print("FILTERING OSQ DATA")
    print("=" * 80)
    
    original_count = len(osq_df)
    
    # Normalize FILTER_JUDGE to a list
    judge_list = [FILTER_JUDGE] if isinstance(FILTER_JUDGE, str) else FILTER_JUDGE
    
    # Apply filters to osq_df
    mask = pd.Series([True] * len(osq_df))
    
    if judge_list is not None:
        mask &= osq_df['judge_model'].isin(judge_list)
    if FILTER_PROMPT is not None:
        mask &= osq_df['prompt_id'] == FILTER_PROMPT
    
    osq_filtered = osq_df[mask].copy()
    
    print(f"\nFiltering criteria:")
    print(f"   Judge(s): {judge_list if judge_list else 'All'}")
    print(f"   Prompt: {FILTER_PROMPT if FILTER_PROMPT else 'All'}")
    print(f"\n✅ OSQ DataFrame filtered: {original_count} → {len(osq_filtered)} samples")
else:
    osq_filtered = osq_df.copy() if 'osq_df' in locals() else None
    print("\n📌 No filtering applied. Using all OSQ data.")


FILTERING OSQ DATA

Filtering criteria:
   Judge(s): ['openai_gpt-5-mini', 'gpt-oss_120b']
   Prompt: All

✅ OSQ DataFrame filtered: 40960 → 32110 samples


In [11]:
osq_filtered

,model,question_id,category,tags,osq_prompt,expected_answer,model_response,blooms_level,judge_model,prompt_id,...,completeness,clarity_organization,professional_relevance,total_score,percentage,is_correct,parse_status,has_response,sample_id,file
0,anthropic__claude-sonnet-4.5,1,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,"Define ""uncertainty"" in systems engineering. A...",A condition in which system outcomes or behavi...,[Uncertainty in systems engineering refers to ...,Remember,gpt-oss_120b,p1,...,20.0,18.0,18.0,93.0,93.0,True,success,True,0,samples_sysengbench-osq_2025-11-16T21-02-49.45...
1,anthropic__claude-sonnet-4.5,2,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,"In systems engineering, define ""risk"" precisel...",Risk is the potential for loss or an undesirab...,"[In systems engineering, risk represents the p...",Remember,gpt-oss_120b,p1,...,20.0,20.0,19.0,98.0,98.0,True,success,True,1,samples_sysengbench-osq_2025-11-16T21-02-49.45...
2,anthropic__claude-sonnet-4.5,3,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,"In systems engineering, identify the two main ...",Epistemic uncertainty: reducible uncertainty d...,[The two main types of uncertainty in systems ...,Remember,gpt-oss_120b,p1,...,20.0,20.0,19.0,98.0,98.0,True,success,True,2,samples_sysengbench-osq_2025-11-16T21-02-49.45...
3,anthropic__claude-sonnet-4.5,4,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,"In systems engineering, what is the primary go...",To ensure that all system components function ...,[The primary goal of systems integration is to...,Remember,gpt-oss_120b,p1,...,19.0,19.0,17.0,93.0,93.0,True,success,True,3,samples_sysengbench-osq_2025-11-16T21-02-49.45...
4,anthropic__claude-sonnet-4.5,5,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,"In systems engineering, define ""reliability."" ...",The ability to perform required functions unde...,[Reliability is the probability that a system ...,Remember,gpt-oss_120b,p1,...,20.0,18.0,18.0,95.0,95.0,True,success,True,4,samples_sysengbench-osq_2025-11-16T21-02-49.45...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40755,phi4__14b,1140,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,"In MIL-STD-461, which specific test assesses e...",RE102,[The specific test in MIL-STD-461 that assesse...,Remember,openai_gpt-5-mini,p1,...,3.0,14.0,6.0,31.0,31.0,False,success,True,840,samples_sysengbench-osq_2025-11-14T23-10-44.86...
40756,phi4__14b,1141,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,"In MIL-STD-461, which specific test evaluates ...",RE103,[The specific test in MIL-STD-461 that evaluat...,Remember,openai_gpt-5-mini,p1,...,5.0,16.0,8.0,39.0,39.0,False,success,True,841,samples_sysengbench-osq_2025-11-14T23-10-44.86...
40757,phi4__14b,1142,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,"In MIL-STD-461, which specific test evaluates ...",RS101,[The specific test in MIL-STD-461 that evaluat...,Remember,openai_gpt-5-mini,p1,...,10.0,16.0,12.0,67.0,67.0,False,success,True,842,samples_sysengbench-osq_2025-11-14T23-10-44.86...
40758,phi4__14b,1143,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,"In MIL-STD-461, which test evaluates equipment...",RS103,[The test in MIL-STD-461 that evaluates equipm...,Remember,openai_gpt-5-mini,p1,...,12.0,18.0,15.0,74.0,74.0,True,success,True,843,samples_sysengbench-osq_2025-11-14T23-10-44.86...


#### Create the aligned_df (mcq_df + osq_filtered)
`osq_filtered` is sometimes simply the same as osq_df if we did not filter out any judges.

In [12]:
# Use existing osq_filtered data
print("=" * 80)
print("OSQ DATA SUMMARY")
print("=" * 80)

if 'osq_filtered' in locals() and osq_filtered is not None and len(osq_filtered) > 0:
    print(f"\n✅ Using osq_filtered DataFrame: {osq_filtered.shape}")
    print(f"   Models: {osq_filtered['model'].unique().tolist()}")
    print(f"   Questions: {osq_filtered['question_id'].nunique()}")
    print(f"   Mean score: {osq_filtered['total_score'].mean():.2f}")
    
    # Count samples by judge and prompt
    print("\n📊 Sample counts by judge and prompt:")
    judge_prompt_counts = osq_filtered.groupby(['judge_model', 'prompt_id']).size().reset_index(name='count')
    for _, row in judge_prompt_counts.iterrows():
        print(f"   {row['judge_model']:30s} | {row['prompt_id']:5s} | {row['count']:4d} samples")
    
    # Align MCQ and OSQ data
    print("\n📊 Aligning MCQ and OSQ data...")
    aligned_df = pd.DataFrame(align_mcq_osq_results(mcq_df, osq_filtered.to_dict('records')))
    print(f"✅ Aligned DataFrame shape: {aligned_df.shape}")
    
    osq_available = True
else:
    print("⚠️  No OSQ filtered data available. OSQ plots will be skipped.")
    osq_available = False
    aligned_df = pd.DataFrame()

OSQ DATA SUMMARY

✅ Using osq_filtered DataFrame: (32110, 22)
   Models: ['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b', 'llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']
   Questions: 845
   Mean score: 76.30

📊 Sample counts by judge and prompt:
   gpt-oss_120b                   | p1    | 16055 samples
   openai_gpt-5-mini              | p1    | 16055 samples

📊 Aligning MCQ and OSQ data...
✅ Aligned 43472 records across 2 judge(s) (strategy: drop)
✅ Aligned DataFrame shape: (43472, 24)


In [18]:
# Use existing osq_filtered data
print("=" * 80)
print("OSQ DATA SUMMARY & ALIGNMENT")
print("=" * 80)

if 'osq_filtered' in locals() and osq_filtered is not None and len(osq_filtered) > 0:
    # Input summary
    print(f"\n📂 INPUT: osq_filtered")
    print(f"   Total samples: {len(osq_filtered):,}")
    print(f"   Unique models: {osq_filtered['model'].nunique()}")
    print(f"   Unique questions: {osq_filtered['question_id'].nunique()}")
    print(f"   Judges: {osq_filtered['judge_model'].nunique()}")
    print(f"   Models: {osq_filtered['model'].unique().tolist()}")
    
    # Sample counts by judge
    print("\n📊 Samples per judge:")
    for judge, count in osq_filtered.groupby('judge_model').size().items():
        print(f"   {judge:30s}: {count:,} samples")
    
    # Alignment
    print("\n" + "-" * 40)
    print("🔗 ALIGNING MCQ + OSQ...")
    print(f"   MCQ input: {len(mcq_df):,} samples ({mcq_df['question_id'].nunique()} questions)")
    print(f"   OSQ input: {len(osq_filtered):,} samples ({osq_filtered['question_id'].nunique()} questions)")
    
    aligned_df = pd.DataFrame(align_mcq_osq_results(mcq_df, osq_filtered.to_dict('records')))
    
    print(f"\n✅ OUTPUT: aligned_df")
    print(f"   Total aligned samples: {len(aligned_df):,}")
    if len(aligned_df) > 0:
        print(f"   Unique models: {aligned_df['model'].nunique()}")
        print(f"   Unique questions: {aligned_df['question_id'].nunique()}")
    
    osq_available = True
else:
    print("\n⚠️  No OSQ filtered data available. OSQ analyses will be skipped.")
    osq_available = False
    aligned_df = pd.DataFrame()

print("\n" + "=" * 80)

OSQ DATA SUMMARY & ALIGNMENT

📂 INPUT: osq_filtered
   Total samples: 32,110
   Unique models: 19
   Unique questions: 845
   Judges: 2
   Models: ['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b', 'llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']

📊 Samples per judge:
   gpt-oss_120b                  : 16,055 samples
   openai_gpt-5-mini             : 16,055 samples

----------------------------------------
🔗 ALIGNING MCQ + OSQ...
   MCQ input: 108,680 samples (1144 questions)
   OSQ input: 32,110 samples (845 questions)
✅ Aligned 43472 records across 2 judge(s) (strategy: drop)

✅ OUTPUT: aligned_df
   Total aligned samples: 43,472
   Unique models: 19
   Unique questions: 1144



In [13]:
aligned_df

,model,question_id,category,tags,question,mcq_random_correct,mcq_a_correct,mcq_b_correct,mcq_c_correct,mcq_d_correct,...,osq_technical_accuracy,osq_conceptual_understanding,osq_completeness,osq_clarity_organization,osq_professional_relevance,blooms_level,judge_model,prompt_id,parse_status,has_response
0,anthropic__claude-sonnet-4.5,1,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,What best describes the concept of uncertainty...,True,True,True,True,True,...,20.0,20.0,20.0,20.0,20.0,Remember,openai_gpt-5-mini,p1,success,True
1,anthropic__claude-sonnet-4.5,1,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,What best describes the concept of uncertainty...,True,True,True,True,True,...,18.0,19.0,20.0,18.0,18.0,Remember,gpt-oss_120b,p1,success,True
2,anthropic__claude-sonnet-4.5,2,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,How is risk defined in systems engineering?,True,True,True,True,True,...,20.0,20.0,20.0,20.0,20.0,Remember,openai_gpt-5-mini,p1,success,True
3,anthropic__claude-sonnet-4.5,2,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,How is risk defined in systems engineering?,True,True,True,True,True,...,20.0,19.0,20.0,20.0,19.0,Remember,gpt-oss_120b,p1,success,True
4,anthropic__claude-sonnet-4.5,3,INCOSEHandbook/Systems Engineering Overview/Sy...,Introduction to risk,Which of the following best describes the two ...,True,True,True,True,True,...,20.0,20.0,20.0,20.0,20.0,Remember,openai_gpt-5-mini,p1,success,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43467,phi4__14b,1142,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,Which test within MIL-STD-461 evaluates equipm...,True,False,True,False,True,...,8.0,18.0,12.0,16.0,14.0,Remember,gpt-oss_120b,p1,success,True
43468,phi4__14b,1143,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,Which test within MIL-STD-461 evaluates equipm...,False,False,True,True,True,...,12.0,17.0,12.0,18.0,15.0,Remember,openai_gpt-5-mini,p1,success,True
43469,phi4__14b,1143,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,Which test within MIL-STD-461 evaluates equipm...,False,False,True,True,True,...,5.0,15.0,8.0,18.0,14.0,Remember,gpt-oss_120b,p1,success,True
43470,phi4__14b,1144,INCOSEHandbook/Specialty Engineering Activitie...,Design Standards,Which test within MIL-STD-461 evaluates equipm...,False,False,True,False,False,...,5.0,12.0,3.0,17.0,10.0,Remember,openai_gpt-5-mini,p1,success,True


#### Understanding `aligned_df`

The `aligned_df` DataFrame **aligns MCQ and OSQ data by (model, question_id, judge_model)**. It contains:

- **All MCQ questions**: Every question from `mcq_df`, with OSQ scores added where available
- **One row per (model, question_id, judge_model)**: Each row represents a single model's response to a single question, for each judge
- **MCQ results across all position variants**: `mcq_a_correct`, `mcq_b_correct`, `mcq_c_correct`, `mcq_d_correct`, plus `mcq_avg_correct`
- **OSQ rubric scores**: `osq_total_score`, `osq_percentage`, and individual dimension scores (may be `None` if no OSQ match)
- **Metadata**: `category`, `tags`, `blooms_level`, `parse_status`

**Key columns:**
| Column | Description |
|--------|-------------|
| `model` | Model under evaluation |
| `question_id` | Unique question identifier |
| `judge_model` | LLM judge that scored the OSQ response |
| `mcq_avg_correct` | Average correctness across position variants (0-1) |
| `osq_total_score` | Total OSQ score (0-100), or `None` if no OSQ data |
| `osq_percentage` | OSQ score as percentage |
| `parse_status` | `success`, `model_error`, `judge_error`, `api_error`, or `None` |

⚠️ **Note**: For MCQ vs OSQ comparison analyses, you must **filter to the intersection** — rows where both MCQ and OSQ data exist (i.e., `osq_total_score.notna()`). This ensures fair comparison on the same question set.

#### aligned_df_intersection (OSQ mapped to equivalent MCQ)

In [19]:
# Create aligned_df_intersection for MCQ vs OSQ comparison analysis
print("=" * 80)
print("CREATING ALIGNED INTERSECTION FOR MCQ vs OSQ COMPARISON")
print("=" * 80)

# Filter to rows where both MCQ and OSQ data exist
aligned_df_intersection = aligned_df[
    (aligned_df['mcq_avg_correct'].notna()) & 
    (aligned_df['osq_total_score'].notna())
].copy()

print(f"\n📂 INPUT: aligned_df")
print(f"   Total rows: {len(aligned_df):,}")
print(f"   Unique questions: {aligned_df['question_id'].nunique()}")

print(f"\n✅ OUTPUT: aligned_df_intersection")
print(f"   Total rows: {len(aligned_df_intersection):,}")
print(f"   Unique models: {aligned_df_intersection['model'].nunique()}")
print(f"   Unique questions: {aligned_df_intersection['question_id'].nunique()}")
print(f"   Unique judges: {aligned_df_intersection['judge_model'].nunique()}")

# Show breakdown
print(f"\n📊 Rows per judge:")
for judge, count in aligned_df_intersection.groupby('judge_model').size().items():
    print(f"   {judge:30s}: {count:,} rows")

# Calculate how much was dropped
dropped = len(aligned_df) - len(aligned_df_intersection)
drop_pct = (dropped / len(aligned_df)) * 100 if len(aligned_df) > 0 else 0
print(f"\n📉 Dropped {dropped:,} rows ({drop_pct:.1f}%) missing MCQ or OSQ data")

print("\n" + "=" * 80)

CREATING ALIGNED INTERSECTION FOR MCQ vs OSQ COMPARISON

📂 INPUT: aligned_df
   Total rows: 43,472
   Unique questions: 1144

✅ OUTPUT: aligned_df_intersection
   Total rows: 32,105
   Unique models: 19
   Unique questions: 845
   Unique judges: 2

📊 Rows per judge:
   gpt-oss_120b                  : 16,051 rows
   openai_gpt-5-mini             : 16,054 rows

📉 Dropped 11,367 rows (26.1%) missing MCQ or OSQ data



### 1.5 Data Summary

#### Parse Status Attribution

The OSQ parser now retains ALL samples and tracks their `parse_status`:

- **`success`**: Judge successfully extracted all 5 scores from model response → normal scoring
- **`model_error`**: Model provided empty/whitespace-only response → assign `total_score=0`, `is_correct=False`
- **`judge_error`**: Model provided non-empty response but judge failed to extract scores → assign `total_score=None` (exclude from metrics)
- **`api_error`**: Judge API failure (timeout, connection reset, etc.) → assign `total_score=None` (exclude from metrics)

**Rationale**: Any non-empty response deserves grading. If the model tried but failed (empty response), it receives 0 points. If the model submitted a response but the judge malfunctioned, we exclude that sample to avoid penalizing the model unfairly.

In [17]:
# Get unique models (intersection of MCQ and OSQ)
mcq_models = set(mcq_df['model'].unique())
osq_models = set(osq_filtered['model'].unique())
common_models = sorted(list(mcq_models & osq_models))

# Parse status breakdown
if 'parse_status' in osq_df.columns:
    status_counts = osq_df['parse_status'].value_counts()
    status_breakdown = f"\nOSQ Parse Status Breakdown:\n"
    for status, count in status_counts.items():
        pct = (count / len(osq_df)) * 100
        status_breakdown += f"  {status}: {count:,} ({pct:.1f}%)\n"
    print(status_breakdown)
    
    # Error breakdown by model (columns = models, rows = error types)
    error_statuses = ['judge_error', 'api_error', 'model_error']
    error_df = osq_df[osq_df['parse_status'].isin(error_statuses)]
    
    if len(error_df) > 0:
        print('\n' + '='*80)
        print('OSQ PARSE ERRORS BY MODEL UNDER EVALUATION')
        print('='*80)
        
        error_pivot = pd.crosstab(
            error_df['parse_status'], 
            error_df['model'],
            margins=True, 
            margins_name='Total'
        )
        
        row_order = [s for s in error_statuses if s in error_pivot.index] + ['Total']
        error_pivot = error_pivot.reindex(row_order)
        print(error_pivot.to_string())
        print()
        
        print('\n' + '='*80)
        print('OSQ PARSE ERRORS BY JUDGE MODEL')
        print('='*80)
        
        judge_error_pivot = pd.crosstab(
            error_df['parse_status'],
            error_df['judge_model'],
            margins=True,
            margins_name='Total'
        )
        judge_error_pivot = judge_error_pivot.reindex(row_order)
        print(judge_error_pivot.to_string())
        print()

# Calculate matched question IDs (intersection of MCQ and OSQ questions)
mcq_question_ids = set(mcq_df['question_id'].unique())
osq_question_ids = set(osq_filtered['question_id'].unique())
matched_question_ids = mcq_question_ids & osq_question_ids

# Identify complete judges if not already defined
if 'COMPLETE_JUDGES' not in locals():
    COMPLETE_JUDGES = identify_complete_judges(osq_filtered, n_required_questions=len(matched_question_ids))

summary_data = {
    'Metric': [
        'Total MCQ samples',
        'Total OSQ samples',
        'OSQ filtered (by judge)',
        'OSQ success samples',
        'OSQ model_error samples',
        'OSQ judge_error samples',
        'OSQ api_error samples',
        'Unique MCQ questions',
        'Unique OSQ questions',
        'Matched questions',
        'MCQ models',
        'OSQ models',
        'Common models',
        'Complete judges',
        'Position variants'
    ],
    'Count': [
        len(mcq_df),
        len(osq_df),
        len(osq_filtered),
        len(osq_df[osq_df['parse_status'] == 'success']) if 'parse_status' in osq_df.columns else 0,
        len(osq_df[osq_df['parse_status'] == 'model_error']) if 'parse_status' in osq_df.columns else 0,
        len(osq_df[osq_df['parse_status'] == 'judge_error']) if 'parse_status' in osq_df.columns else 0,
        len(osq_df[osq_df['parse_status'] == 'api_error']) if 'parse_status' in osq_df.columns else 0,
        mcq_df['question_id'].nunique(),
        osq_df['question_id'].nunique(),
        len(matched_question_ids),
        len(mcq_models),
        len(osq_models),
        len(common_models),
        len(COMPLETE_JUDGES),
        len(POSITION_MAPPING)
    ]
}

summary_df = pd.DataFrame(summary_data)
print('\n' + '='*60)
print('DATA SUMMARY')
print('='*60)
print(summary_df.to_string(index=False))
print('='*60)


OSQ Parse Status Breakdown:
  success: 40,933 (99.9%)
  api_error: 19 (0.0%)
  judge_error: 8 (0.0%)


OSQ PARSE ERRORS BY MODEL UNDER EVALUATION
model         anthropic__claude-sonnet-4.5  devstral__24b  gemma3__1b  gemma3__27b  gemma3__4b  google__gemini-2.5-flash  llama3.2__1b  llama3.2__3b  llama3.3__70b  Total
parse_status                                                                                                                                                              
judge_error                              4              0           1            0           0                         2             1             0              0      8
api_error                                6              3           0            2           1                         1             0             5              1     19
Total                                   10              3           1            2           1                         3             1             5              1     2

In [27]:
# =============================================================================
# 1. MCQ DATA SUMMARY
# =============================================================================
print("=" * 80)
print("1. MCQ DATA SUMMARY")
print("=" * 80)

print(f"\n📂 Source: mcq_df")
print(f"   Total samples: {len(mcq_df):,}")
print(f"   Unique models: {mcq_df['model'].nunique()}")
print(f"   Unique questions: {mcq_df['question_id'].nunique()}")
print(f"   Variants: {sorted(mcq_df['variant'].unique())}")

# Samples per variant
print(f"\n📊 Samples per variant:")
for variant, count in mcq_df.groupby('variant').size().items():
    print(f"   {variant}: {count:,}")

# Models list
print(f"\n📋 Models: {sorted(mcq_df['model'].unique())}")

# =============================================================================
# 2. OSQ DATA SUMMARY
# =============================================================================
print("\n" + "=" * 80)
print("2. OSQ DATA SUMMARY")
print("=" * 80)

print(f"\n📂 Source: osq_df (full) → osq_filtered (after judge filter)")
print(f"   osq_df total: {len(osq_df):,}")
print(f"   osq_filtered total: {len(osq_filtered):,}")
print(f"   Unique models: {osq_filtered['model'].nunique()}")
print(f"   Unique questions: {osq_filtered['question_id'].nunique()}")
print(f"   Unique judges: {osq_filtered['judge_model'].nunique()}")

# Parse status breakdown
if 'parse_status' in osq_filtered.columns:
    print(f"\n📋 Parse Status Breakdown:")
    status_counts = osq_filtered['parse_status'].value_counts()
    for status, count in status_counts.items():
        pct = (count / len(osq_filtered)) * 100
        print(f"   {status:15s}: {count:,} ({pct:.1f}%)")
    
    # Error breakdown by model
    error_statuses = ['judge_error', 'api_error', 'model_error']
    error_df = osq_filtered[osq_filtered['parse_status'].isin(error_statuses)]
    
    if len(error_df) > 0:
        print('\n' + '-'*60)
        print('OSQ PARSE ERRORS BY MODEL')
        print('-'*60)
        error_pivot = pd.crosstab(
            error_df['parse_status'], 
            error_df['model'],
            margins=True, 
            margins_name='Total'
        )
        row_order = [s for s in error_statuses if s in error_pivot.index] + ['Total']
        error_pivot = error_pivot.reindex(row_order)
        print(error_pivot.to_string())
        
        print('\n' + '-'*60)
        print('OSQ PARSE ERRORS BY JUDGE')
        print('-'*60)
        judge_error_pivot = pd.crosstab(
            error_df['parse_status'],
            error_df['judge_model'],
            margins=True,
            margins_name='Total'
        )
        judge_error_pivot = judge_error_pivot.reindex(row_order)
        print(judge_error_pivot.to_string())

# Judges list
print(f"\n📋 Judges: {sorted(osq_filtered['judge_model'].unique())}")

# =============================================================================
# 3. ALIGNMENT SUMMARY (MCQ vs OSQ Comparison)
# =============================================================================
print("\n" + "=" * 80)
print("3. ALIGNMENT SUMMARY (MCQ vs OSQ)")
print("=" * 80)

print(f"\n📂 aligned_df (full alignment)")
print(f"   Total rows: {len(aligned_df):,}")
print(f"   Unique models: {aligned_df['model'].nunique()}")
print(f"   Unique questions: {aligned_df['question_id'].nunique()}")
print(f"   Unique judges: {aligned_df['judge_model'].nunique()}")

# How many have MCQ only, OSQ only, or both
mcq_only = aligned_df['osq_total_score'].isna().sum()
has_both = aligned_df['osq_total_score'].notna().sum()
print(f"\n📊 Coverage:")
print(f"   Rows with OSQ data: {has_both:,}")
print(f"   Rows without OSQ (MCQ-only): {mcq_only:,}")

print(f"\n📂 aligned_df_intersection (MCQ+OSQ matched)")
print(f"   Total rows: {len(aligned_df_intersection):,}")
print(f"   Unique models: {aligned_df_intersection['model'].nunique()}")
print(f"   Unique questions: {aligned_df_intersection['question_id'].nunique()}")
print(f"   Unique judges: {aligned_df_intersection['judge_model'].nunique()}")

# Set COMPLETE_JUDGES for downstream analysis
COMPLETE_JUDGES = sorted(aligned_df_intersection['judge_model'].unique())
print(f"\n📋 Judges for comparison analysis: {COMPLETE_JUDGES}")

# Samples per judge in intersection
print(f"\n📊 Intersection samples per judge:")
for judge, count in aligned_df_intersection.groupby('judge_model').size().items():
    print(f"   {judge:30s}: {count:,}")

print("\n" + "=" * 80)
print("✅ Data ready for analysis")
print("=" * 80)

1. MCQ DATA SUMMARY

📂 Source: mcq_df
   Total samples: 108,680
   Unique models: 19
   Unique questions: 1144
   Variants: ['a', 'b', 'c', 'd', 'random']

📊 Samples per variant:
   a: 21,736
   b: 21,736
   c: 21,736
   d: 21,736
   random: 21,736

📋 Models: ['anthropic__claude-sonnet-4.5', 'devstral__24b', 'gemma3__12b', 'gemma3__1b', 'gemma3__27b', 'gemma3__4b', 'google__gemini-2.5-flash', 'llama3.2__1b', 'llama3.2__3b', 'llama3.3__70b', 'llama4__16x17b', 'mistral-large__123b', 'mistral-small3.2__24b', 'mixtral__8x22b', 'openai__gpt-4.1', 'phi3.5__3.8b', 'phi3__14b', 'phi4-mini__3.8b', 'phi4__14b']

2. OSQ DATA SUMMARY

📂 Source: osq_df (full) → osq_filtered (after judge filter)
   osq_df total: 40,960
   osq_filtered total: 32,110
   Unique models: 19
   Unique questions: 845
   Unique judges: 2

📋 Parse Status Breakdown:
   success        : 32,105 (100.0%)
   judge_error    : 3 (0.0%)
   api_error      : 2 (0.0%)

------------------------------------------------------------
OSQ PA

#### Dataframes passed on for Analyses in Section 2, 3, and 4.

Data ready for analysis:
- MCQ Position Bias: use `mcq_df` or `aligned_df` with filtering to drop non-relevant OSQ columns
- OSQ Analysis: use `osq_filtered` or `aligned_df` or `aligned_df_intersection`, with filtering i fusing the latter two to drop non-relevant MCQ columns
- MCQ vs OSQ Comparison: use `aligned_df_intersection`

The analysis will use:
- MCQ Position Bias: use `mcq_df`
  - This allows us to use ALL of the MCQs, and not just the ones that align to OSQ. It also means we do not have to filter.
- OSQ Analysis: use `osq_filtered` or `aligned_df` or `aligned_df_intersection`, with filtering on latter two to drop non-relevant MCQ columns
  - Using `aligned_df_intersection` for commonality with next section.
- MCQ vs OSQ Comparison: use `aligned_df_intersection`
  - The only dataset that has the 1:1 MCQ vs OSQ.
- Cost Analysis: use `osq_filtered` 
  - Using `osq_filtered` as it has the raw OSQ token outputs for tokenomics 


---
## 2. MCQ POSITION BIAS ANALYSIS (§ 3.5.1)

Position bias refers to systematic preference for particular answer positions (A, B, C, D) independent of question content.

### Analyses:
1. **Visual Diagnostics:** Accuracy by position, deviation from uniform
2. **Chi-square test:** Test independence of position and correctness
3. **Friedman test:** Repeated measures across positions
4. **Effect size:** Cramér's V
5. **Interpretation:** Apply manuscript decision logic

### 2.1 Calculate Per-Position Accuracy

In [ ]:
print('\n📊 Calculating per-position accuracy...\n')

# Accuracy by position and model
position_accuracy = mcq_position.groupby(['model', 'position'])['is_correct'].agg([
    ('accuracy', 'mean'),
    ('n_correct', 'sum'),
    ('n_total', 'count')
]).reset_index()

# Pivot for visualization
acc_pivot = position_accuracy.pivot(
    index='model',
    columns='position',
    values='accuracy'
)

print('Accuracy by Position (first 5 models):')
print(acc_pivot.head())

# Overall position statistics
overall_position = mcq_position.groupby('position')['is_correct'].agg([
    ('mean', 'mean'),
    ('std', 'std'),
    ('count', 'count')
])

print('\nOverall Position Statistics:')
print(overall_position)

### 2.2 Visual Diagnostic: Accuracy by Position

In [ ]:
print('\n📊 PLOT 1: Accuracy by Position\n')

fig, ax = plt.subplots(figsize=(14, 10))

# Grouped bar chart
acc_pivot.plot(
    kind='bar',
    ax=ax,
    width=0.8,
    edgecolor='black',
    linewidth=0.5
)

ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('MCQ Accuracy by Answer Position (A/B/C/D)', 
             fontsize=14, fontweight='bold')
ax.legend(title='Position', fontsize=10)
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(output_dir / 'fig_mcq_position_accuracy.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✅ Saved: {output_dir}/fig_mcq_position_accuracy.png')

### 2.3 Visual Diagnostic: Deviation from Uniform

In [ ]:
print('\n📊 PLOT 2: Deviation from Uniform\n')

# Calculate deviation from uniform baseline
# Δ_p = Acc_p - (1/4)Σ Acc_i
uniform_baseline = acc_pivot.mean(axis=1)
deviation = acc_pivot.subtract(uniform_baseline, axis=0)

print('Deviation from Uniform (first 5 models):')
print(deviation.head())

fig, ax = plt.subplots(figsize=(14, 10))

# Grouped bar chart showing deviations
deviation.plot(
    kind='bar',
    ax=ax,
    width=0.8,
    edgecolor='black',
    linewidth=0.5,
    colormap='RdYlGn'
)

ax.axhline(0, color='black', linestyle='-', linewidth=1, alpha=0.7)
ax.set_xlabel('Model', fontsize=12)
ax.set_ylabel('Deviation from Mean Accuracy', fontsize=12)
ax.set_title('MCQ Position Bias: Deviation from Uniform Performance', 
             fontsize=14, fontweight='bold')
ax.legend(title='Position', fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
plt.savefig(output_dir / 'fig_mcq_deviation_uniform.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✅ Saved: {output_dir}/fig_mcq_deviation_uniform.png')

### 2.4 Statistical Test: Chi-Square Test of Independence

In [ ]:
print('\n📊 Chi-Square Test of Independence\n')
print('H₀: Answer position does not affect correctness\n')

chi_square_results = []

for model in common_models:
    model_data = mcq_position[mcq_position['model'] == model]
    
    # Create 2×4 contingency table
    # Rows: correct (True/False)
    # Columns: positions (A/B/C/D)
    contingency = pd.crosstab(
        model_data['is_correct'],
        model_data['position']
    )
    
    # Chi-square test
    chi2_stat, p_value, dof, expected = chi2_contingency(contingency)
    
    # Cramér's V (effect size)
    n = contingency.sum().sum()
    min_dim = min(contingency.shape[0] - 1, contingency.shape[1] - 1)
    cramers_v = np.sqrt(chi2_stat / (n * min_dim))
    
    chi_square_results.append({
        'model': model,
        'chi2_stat': chi2_stat,
        'p_value': p_value,
        'dof': dof,
        'cramers_v': cramers_v,
        'n_observations': int(n),
        'significant': 'Yes' if p_value < 0.05 else 'No'
    })

chi_square_df = pd.DataFrame(chi_square_results)

print('Chi-Square Test Results (first 10 models):')
print(chi_square_df.head(10).to_string(index=False))

# Summary statistics
n_significant = (chi_square_df['p_value'] < 0.05).sum()
print(f'\nSignificant results (p < 0.05): {n_significant}/{len(chi_square_df)}')
print(f"Mean Cramér's V: {chi_square_df['cramers_v'].mean():.4f}")
print(f"Median Cramér's V: {chi_square_df['cramers_v'].median():.4f}")

In [ ]:
# Export chi-square results
chi_square_df.to_csv(output_dir / 'table_chi_square_position_bias.csv', index=False)
export_latex_table(
    chi_square_df[['model', 'chi2_stat', 'p_value', 'cramers_v', 'significant']],
    'table_chi_square_position_bias.tex',
    'Chi-Square Test Results for Position Bias',
    'tab:chi_square_position'
)

### 2.5 Statistical Test: Friedman Test (Repeated Measures)

In [ ]:
print('\n📊 Friedman Test (Repeated Measures)\n')
print('H₀: Per-question correctness does not differ across positions\n')

friedman_results = []

for model in common_models:
    model_data = mcq_position[mcq_position['model'] == model]
    
    # Pivot to repeated measures format
    # Rows: questions, Columns: positions (A/B/C/D)
    repeated_measures = model_data.pivot(
        index='question_id',
        columns='position',
        values='is_correct'
    )
    
    # Drop questions with missing positions (if any)
    repeated_measures = repeated_measures.dropna()
    
    if len(repeated_measures) > 0:
        # Friedman test requires arrays for each condition
        position_arrays = [
            repeated_measures['A'].values,
            repeated_measures['B'].values,
            repeated_measures['C'].values,
            repeated_measures['D'].values
        ]
        
        # Friedman test
        friedman_stat, p_value = friedmanchisquare(*position_arrays)
        
        # Kendall's W (effect size for Friedman)
        n_subjects = len(repeated_measures)
        k_conditions = 4
        kendalls_w = friedman_stat / (n_subjects * (k_conditions - 1))
        
        friedman_results.append({
            'model': model,
            'friedman_stat': friedman_stat,
            'p_value': p_value,
            'kendalls_w': kendalls_w,
            'n_questions': int(n_subjects),
            'significant': 'Yes' if p_value < 0.05 else 'No'
        })

friedman_df = pd.DataFrame(friedman_results)

print('Friedman Test Results (first 10 models):')
print(friedman_df.head(10).to_string(index=False))

# Summary statistics
n_significant = (friedman_df['p_value'] < 0.05).sum()
print(f'\nSignificant results (p < 0.05): {n_significant}/{len(friedman_df)}')
print(f"Mean Kendall's W: {friedman_df['kendalls_w'].mean():.4f}")
print(f"Median Kendall's W: {friedman_df['kendalls_w'].median():.4f}")

In [ ]:
# Export Friedman results
friedman_df.to_csv(output_dir / 'table_friedman_position_bias.csv', index=False)
export_latex_table(
    friedman_df[['model', 'friedman_stat', 'p_value', 'kendalls_w', 'significant']],
    'table_friedman_position_bias.tex',
    'Friedman Test Results for Position Bias',
    'tab:friedman_position'
)

### 2.6 Effect Size Visualization: Cramér's V

In [ ]:
print("\n📊 PLOT 3: Cramér's V Effect Sizes\n")

fig, ax = plt.subplots(figsize=(12, 10))

# Sort by Cramér's V
chi_square_df_sorted = chi_square_df.sort_values('cramers_v', ascending=True)

bars = ax.barh(
    chi_square_df_sorted['model'],
    chi_square_df_sorted['cramers_v'],
    edgecolor='black',
    linewidth=0.5
)

# Color bars by effect size magnitude
colors = []
for v in chi_square_df_sorted['cramers_v']:
    if v < 0.10:
        colors.append('green')      # Negligible
    elif v < 0.30:
        colors.append('orange')     # Moderate
    else:
        colors.append('red')        # Strong

for bar, color in zip(bars, colors):
    bar.set_color(color)

# Reference lines
ax.axvline(0.10, color='orange', linestyle='--', alpha=0.5, label='V=0.10 (Small)')
ax.axvline(0.30, color='red', linestyle='--', alpha=0.5, label='V=0.30 (Medium)')

ax.set_xlabel("Cramér's V (Effect Size)", fontsize=12)
ax.set_ylabel('Model', fontsize=12)
ax.set_title("Position Bias Effect Sizes (Cramér's V)", 
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'fig_cramers_v_position_bias.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✅ Saved: {output_dir}/fig_cramers_v_position_bias.png')

### 2.7 Interpretation (Manuscript § 3.5.1 Decision Logic)

In [ ]:
def interpret_position_bias(chi2_p, cramers_v, friedman_p):
    """
    Apply manuscript interpretation logic for position bias.
    
    Parameters:
    -----------
    chi2_p : float
        Chi-square p-value
    cramers_v : float
        Cramér's V effect size
    friedman_p : float
        Friedman test p-value
    
    Returns:
    --------
    tuple : (category, description)
    """
    # Step 1: Check chi-square significance
    chi2_sig = chi2_p < 0.05
    
    # Step 2: Assess magnitude (Cramér's V)
    if cramers_v < 0.10:
        v_interp = 'negligible'
    elif cramers_v < 0.30:
        v_interp = 'moderate'
    else:
        v_interp = 'strong'
    
    # Step 3: Check Friedman significance
    friedman_sig = friedman_p < 0.05
    
    # Step 4: Final interpretation
    if not chi2_sig and cramers_v < 0.10:
        category = 'No bias'
        description = 'MCQ accuracy is position-invariant'
    elif chi2_sig and cramers_v < 0.10 and not friedman_sig:
        category = 'Mild bias'
        description = 'Statistically significant but small effect'
    elif chi2_sig and 0.10 <= cramers_v < 0.30:
        category = 'Moderate bias'
        description = 'Position should be considered in interpretation'
    elif chi2_sig and cramers_v >= 0.30 and friedman_sig:
        category = 'Strong bias'
        description = 'MCQ accuracy heavily confounded by position'
    else:
        category = 'Mixed evidence'
        description = 'Inconsistent evidence across tests'
    
    return category, description


print('\n📋 Position Bias Interpretation\n')

# Apply to all models
interpretations = []
for idx, row in chi_square_df.iterrows():
    friedman_row = friedman_df[friedman_df['model'] == row['model']]
    
    if len(friedman_row) > 0:
        friedman_p = friedman_row['p_value'].values[0]
    else:
        friedman_p = 1.0  # Conservative if missing
    
    category, description = interpret_position_bias(
        row['p_value'],
        row['cramers_v'],
        friedman_p
    )
    
    interpretations.append({
        'model': row['model'],
        'category': category,
        'description': description,
        'chi2_p': row['p_value'],
        'cramers_v': row['cramers_v'],
        'friedman_p': friedman_p
    })

interpretation_df = pd.DataFrame(interpretations)

print('Position Bias Interpretation (first 10 models):')
print(interpretation_df[['model', 'category', 'cramers_v']].head(10).to_string(index=False))

# Summary
print('\nInterpretation Summary:')
print(interpretation_df['category'].value_counts())

In [ ]:
# Export interpretation
interpretation_df.to_csv(output_dir / 'table_position_bias_interpretation.csv', index=False)
export_latex_table(
    interpretation_df[['model', 'category', 'cramers_v', 'chi2_p', 'friedman_p']],
    'table_position_bias_interpretation.tex',
    'Position Bias Interpretation Summary',
    'tab:position_interpretation'
)

## Section OSQ Analysis

### 3.6 Inter-Judge Agreement (Spearman Correlation)

In [ ]:
print('\n📊 Inter-Judge Agreement Analysis\n')

if len(COMPLETE_JUDGES) >= 2:
    # Pivot for judge comparison
    judge_pivot = osq_filtered.pivot_table(
        index=['model', 'question_id'],
        columns='judge_model',
        values='total_score'
    )
    
    # Pairwise correlations
    correlations = []
    for judge1, judge2 in combinations(COMPLETE_JUDGES, 2):
        valid_data = judge_pivot[[judge1, judge2]].dropna()
        
        if len(valid_data) > 0:
            rho, p_value = spearmanr(valid_data[judge1], valid_data[judge2])
            
            correlations.append({
                'judge_1': judge1,
                'judge_2': judge2,
                'spearman_rho': rho,
                'p_value': p_value,
                'n': len(valid_data)
            })
    
    corr_df = pd.DataFrame(correlations)
    
    # Average correlation
    if len(corr_df) > 0:
        avg_rho = corr_df['spearman_rho'].mean()
        print(f'Inter-Judge Correlations:')
        print(corr_df.to_string(index=False))
        print(f'\nAverage Spearman correlation: {avg_rho:.3f}')
        
        # Export
        corr_df.to_csv(output_dir / 'table_interjudge_correlation.csv', index=False)
        export_latex_table(
            corr_df,
            'table_interjudge_correlation.tex',
            'Inter-Judge Agreement (Spearman Correlation)',
            'tab:interjudge_corr'
        )
    else:
        print('Not enough judge pairs for correlation analysis')
else:
    print('Only one judge available - skipping inter-judge correlation')

### 3.7 Bootstrap Confidence Intervals for OSQ Scores

In [ ]:
print('\n📊 Bootstrap Confidence Intervals for OSQ Scores\n')

bootstrap_results_osq = []

for model in common_models:
    model_scores = osq_filtered[osq_filtered['model'] == model]['total_score'].values
    
    if len(model_scores) > 0:
        # Bootstrap using scipy
        rng = np.random.default_rng()
        res = bootstrap(
            (model_scores,),
            np.mean,
            n_resamples=N_BOOTSTRAP,
            confidence_level=CONFIDENCE_LEVEL,
            random_state=rng
        )
        
        bootstrap_results_osq.append({
            'model': model,
            'mean': model_scores.mean(),
            'ci_lower': res.confidence_interval.low,
            'ci_upper': res.confidence_interval.high,
            'ci_width': res.confidence_interval.high - res.confidence_interval.low
        })

bootstrap_osq_df = pd.DataFrame(bootstrap_results_osq)

print('Bootstrap CIs for OSQ (first 10 models):')
print(bootstrap_osq_df.head(10).to_string(index=False))

# Export
bootstrap_osq_df.to_csv(output_dir / 'table_osq_bootstrap_ci.csv', index=False)
export_latex_table(
    bootstrap_osq_df,
    'table_osq_bootstrap_ci.tex',
    'Bootstrap 95% Confidence Intervals for OSQ Scores',
    'tab:osq_bootstrap'
)

### 3.8 Visual: OSQ Bootstrap CI Forest Plot

In [ ]:
print('\n📊 PLOT 7: OSQ Bootstrap Confidence Intervals\n')

bootstrap_osq_sorted = bootstrap_osq_df.sort_values('mean')

fig, ax = plt.subplots(figsize=(10, 12))

y_pos = np.arange(len(bootstrap_osq_sorted))
ax.errorbar(
    bootstrap_osq_sorted['mean'],
    y_pos,
    xerr=[
        bootstrap_osq_sorted['mean'] - bootstrap_osq_sorted['ci_lower'],
        bootstrap_osq_sorted['ci_upper'] - bootstrap_osq_sorted['mean']
    ],
    fmt='o',
    markersize=8,
    capsize=5,
    capthick=2
)

ax.set_yticks(y_pos)
ax.set_yticklabels(bootstrap_osq_sorted['model'])
ax.set_xlabel('Mean OSQ Score (95% Bootstrap CI)', fontsize=12)
ax.set_title('OSQ Performance with Bootstrap Confidence Intervals', 
             fontsize=14, fontweight='bold')
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'fig_osq_bootstrap_ci.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✅ Saved: {output_dir}/fig_osq_bootstrap_ci.png')

### 3.9 OSQ Interpretation

In [ ]:
print('\n📋 OSQ Analysis Interpretation\n')

# Inter-judge agreement interpretation
if 'corr_df' in locals() and len(corr_df) > 0:
    avg_rho = corr_df['spearman_rho'].mean()
    
    if avg_rho >= 0.80:
        judge_interp = 'High reliability: OSQ scores robust to judge choice'
    elif avg_rho >= 0.50:
        judge_interp = 'Moderate reliability: Acknowledge judge effects'
    else:
        judge_interp = 'Low reliability: OSQ conclusions exploratory'
    
    print(f'Inter-Judge Agreement: ρ={avg_rho:.3f}')
    print(f'Interpretation: {judge_interp}')
else:
    print('Inter-judge analysis not available (single judge)')

# Bootstrap CI width interpretation
avg_ci_width = bootstrap_osq_df['ci_width'].mean()
if avg_ci_width < 5.0:
    ci_interp = 'Narrow CIs: Stable mean estimates'
else:
    ci_interp = 'Wide CIs: High uncertainty'

print(f'\nBootstrap CI Width: {avg_ci_width:.2f}')
print(f'Interpretation: {ci_interp}')

---
## 4. FORMAT COMPARISON: MCQ vs OSQ (§ 3.5.3)

Test whether MCQ and OSQ formats produce consistent assessments of model capability.

### Analyses:
1. **Visual Diagnostic:** MCQ-OSQ scatter plot (multi-judge)
2. **Correlation Analysis:** Pearson and Spearman (per judge + average)
3. **Paired Difference Test:** Wilcoxon signed-rank
4. **Effect Size:** Cohen's d
5. **Interpretation:** Apply manuscript decision logic

### 4.1 Prepare Data for Format Comparison

In [ ]:
print('\n📊 Preparing Format Comparison Data\n')

# MCQ accuracy on matched subset (845 questions)
# Use one variant per model (e.g., variant 'a')
mcq_matched_single_variant = mcq_matched[mcq_matched['variant'] == 'a']
mcq_accuracy_matched = mcq_matched_single_variant.groupby('model')['is_correct'].mean()

print(f'MCQ accuracy calculated for {len(mcq_accuracy_matched)} models')
print('\nMCQ Accuracy on Matched Subset (first 10):')
print(mcq_accuracy_matched.head(10))

### 4.2 Visual: MCQ-OSQ Scatter Plot (Multi-Judge)

In [ ]:
print('\n📊 PLOT 8: MCQ vs OSQ Scatter (Multi-Judge)\n')

fig, ax = plt.subplots(figsize=(14, 10))

judge_colors = sns.color_palette('husl', n_colors=len(COMPLETE_JUDGES))

# Plot each judge
for i, judge in enumerate(COMPLETE_JUDGES):
    osq_judge = osq_filtered[osq_filtered['judge_model'] == judge]
    osq_judge_mean = osq_judge.groupby('model')['total_score'].mean()
    
    judge_comparison = pd.DataFrame({
        'mcq': mcq_accuracy_matched,
        'osq': osq_judge_mean / 100  # Normalize to [0,1]
    }).dropna()
    
    # Scatter
    ax.scatter(
        judge_comparison['mcq'],
        judge_comparison['osq'],
        label=judge,
        alpha=0.7,
        s=100,
        color=judge_colors[i]
    )
    
    # Regression line
    if len(judge_comparison) > 1:
        slope, intercept, r, p, se = linregress(
            judge_comparison['mcq'], 
            judge_comparison['osq']
        )
        x_line = np.linspace(
            judge_comparison['mcq'].min(), 
            judge_comparison['mcq'].max(), 
            100
        )
        y_line = slope * x_line + intercept
        ax.plot(x_line, y_line, color=judge_colors[i], 
                linestyle='--', alpha=0.6, linewidth=1.5)

# Average across judges
osq_avg = osq_filtered.groupby('model')['total_score'].mean()
avg_comparison = pd.DataFrame({
    'mcq': mcq_accuracy_matched,
    'osq': osq_avg / 100
}).dropna()

ax.scatter(
    avg_comparison['mcq'],
    avg_comparison['osq'],
    label='Average (All Judges)',
    alpha=0.9,
    s=150,
    color='black',
    marker='D',
    edgecolors='white',
    linewidths=1.5,
    zorder=10
)

# Average regression line
if len(avg_comparison) > 1:
    slope_avg, intercept_avg, r_avg, p_avg, se_avg = linregress(
        avg_comparison['mcq'], 
        avg_comparison['osq']
    )
    x_avg = np.linspace(
        avg_comparison['mcq'].min(), 
        avg_comparison['mcq'].max(), 
        100
    )
    y_avg = slope_avg * x_avg + intercept_avg
    ax.plot(x_avg, y_avg, 'k-', linewidth=3, alpha=0.8, 
            label=f'Avg (r={r_avg:.3f})', zorder=9)

# y=x reference
ax.plot([0, 1], [0, 1], 'k:', linewidth=2, alpha=0.5, label='y=x')

ax.set_xlabel('MCQ Accuracy (845 matched)', fontsize=12)
ax.set_ylabel('OSQ Mean Score (0-1)', fontsize=12)
ax.set_title('Format Comparison: MCQ vs OSQ (Multi-Judge)', 
             fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'fig_mcq_osq_scatter_multijudge.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print(f'✅ Saved: {output_dir}/fig_mcq_osq_scatter_multijudge.png')

### 4.3 Correlation Analysis (Pearson & Spearman)

In [ ]:
print('\n📊 Correlation Analysis (Pearson & Spearman)\n')

correlation_results = []

# Per judge
for judge in COMPLETE_JUDGES:
    osq_judge = osq_filtered[osq_filtered['judge_model'] == judge]
    osq_judge_mean = osq_judge.groupby('model')['total_score'].mean()
    
    comparison = pd.DataFrame({
        'mcq': mcq_accuracy_matched,
        'osq': osq_judge_mean
    }).dropna()
    
    if len(comparison) > 1:
        pearson_r, pearson_p = pearsonr(comparison['mcq'], comparison['osq'])
        spearman_r, spearman_p = spearmanr(comparison['mcq'], comparison['osq'])
        
        correlation_results.append({
            'judge': judge,
            'pearson_r': pearson_r,
            'pearson_p': pearson_p,
            'spearman_r': spearman_r,
            'spearman_p': spearman_p,
            'n': len(comparison)
        })

# Average
osq_avg_mean = osq_filtered.groupby('model')['total_score'].mean()
avg_comp = pd.DataFrame({
    'mcq': mcq_accuracy_matched,
    'osq': osq_avg_mean
}).dropna()

if len(avg_comp) > 1:
    pearson_r_avg, pearson_p_avg = pearsonr(avg_comp['mcq'], avg_comp['osq'])
    spearman_r_avg, spearman_p_avg = spearmanr(avg_comp['mcq'], avg_comp['osq'])
    
    correlation_results.append({
        'judge': 'Average',
        'pearson_r': pearson_r_avg,
        'pearson_p': pearson_p_avg,
        'spearman_r': spearman_r_avg,
        'spearman_p': spearman_p_avg,
        'n': len(avg_comp)
    })

corr_results_df = pd.DataFrame(correlation_results)

print('Correlation Results:')
print(corr_results_df.to_string(index=False))

# Export
corr_results_df.to_csv(output_dir / 'table_mcq_osq_correlations.csv', index=False)
export_latex_table(
    corr_results_df,
    'table_mcq_osq_correlations.tex',
    'MCQ vs OSQ Correlation Analysis',
    'tab:mcq_osq_corr'
)

### 4.4 Wilcoxon Signed-Rank Test

In [ ]:
print('\n📊 Wilcoxon Signed-Rank Test\n')

wilcoxon_results = []

for judge in COMPLETE_JUDGES + ['Average']:
    if judge == 'Average':
        osq_scores = osq_avg_mean / 100
    else:
        osq_scores = osq_filtered[osq_filtered['judge_model'] == judge].groupby('model')['total_score'].mean() / 100
    
    comparison = pd.DataFrame({
        'mcq': mcq_accuracy_matched,
        'osq': osq_scores
    }).dropna()
    
    if len(comparison) > 1:
        differences = comparison['mcq'] - comparison['osq']
        
        stat, p_value = wilcoxon(differences, alternative='two-sided')
        
        wilcoxon_results.append({
            'judge': judge,
            'stat': stat,
            'p_value': p_value,
            'median_diff': differences.median(),
            'mean_diff': differences.mean()
        })

wilcoxon_df = pd.DataFrame(wilcoxon_results)

print('Wilcoxon Test Results:')
print(wilcoxon_df.to_string(index=False))

# Export
wilcoxon_df.to_csv(output_dir / 'table_wilcoxon_results.csv', index=False)
export_latex_table(
    wilcoxon_df,
    'table_wilcoxon_results.tex',
    'Wilcoxon Signed-Rank Test Results',
    'tab:wilcoxon'
)

### 4.5 Effect Size (Cohen's d for Paired Samples)

In [ ]:
print('\n📊 Effect Size (Cohen\'s d)\n')

def cohens_d_paired(differences):
    """Cohen's d for paired samples"""
    return differences.mean() / differences.std(ddof=1)

effect_sizes = []

for judge in COMPLETE_JUDGES + ['Average']:
    if judge == 'Average':
        osq_scores = osq_avg_mean / 100
    else:
        osq_scores = osq_filtered[osq_filtered['judge_model'] == judge].groupby('model')['total_score'].mean() / 100
    
    comparison = pd.DataFrame({
        'mcq': mcq_accuracy_matched,
        'osq': osq_scores
    }).dropna()
    
    if len(comparison) > 1:
        differences = comparison['mcq'] - comparison['osq']
        cohens_d = cohens_d_paired(differences)
        
        if abs(cohens_d) < 0.20:
            interp = 'Negligible'
        elif abs(cohens_d) < 0.50:
            interp = 'Small'
        elif abs(cohens_d) < 0.80:
            interp = 'Medium'
        else:
            interp = 'Large'
        
        effect_sizes.append({
            'judge': judge,
            'cohens_d': cohens_d,
            'interpretation': interp
        })

effect_size_df = pd.DataFrame(effect_sizes)

print('Effect Sizes (Cohen\'s d):')
print(effect_size_df.to_string(index=False))

# Export
effect_size_df.to_csv(output_dir / 'table_effect_sizes.csv', index=False)
export_latex_table(
    effect_size_df,
    'table_effect_sizes.tex',
    'Effect Sizes for Format Comparison',
    'tab:effect_sizes'
)

### 4.6 Per-Model Differences

In [ ]:
print('\n📊 Per-Model Differences (Δᵢ = MCQ - OSQ)\n')

# Use average OSQ across judges
per_model_diff = pd.DataFrame({
    'model': avg_comp.index,
    'mcq_accuracy': avg_comp['mcq'],
    'osq_mean': avg_comp['osq'],
    'delta_i': avg_comp['mcq'] - avg_comp['osq']
}).sort_values('delta_i', ascending=False)

print('Per-Model Differences (first 10):')
print(per_model_diff.head(10).to_string(index=False))

# Export
per_model_diff.to_csv(output_dir / 'table_per_model_differences.csv', index=False)
export_latex_table(
    per_model_diff,
    'table_per_model_differences.tex',
    'Per-Model Format Differences',
    'tab:model_diff'
)

### 4.7 Format Comparison Interpretation

In [ ]:
print('\n📋 Format Comparison Interpretation\n')

# Get average results
avg_pearson = corr_results_df[corr_results_df['judge'] == 'Average']['pearson_r'].values[0]
avg_spearman = corr_results_df[corr_results_df['judge'] == 'Average']['spearman_r'].values[0]
avg_wilcoxon_p = wilcoxon_df[wilcoxon_df['judge'] == 'Average']['p_value'].values[0]
avg_cohens_d = effect_size_df[effect_size_df['judge'] == 'Average']['cohens_d'].values[0]

# 1. Correlations
if avg_pearson >= 0.80 and avg_spearman >= 0.80:
    corr_interp = 'Formats largely equivalent'
elif avg_pearson >= 0.50 or avg_spearman >= 0.50:
    corr_interp = 'Formats moderately aligned'
else:
    corr_interp = 'Formats divergent'

# 2. Wilcoxon
if avg_wilcoxon_p >= 0.05:
    wilcox_interp = 'No systematic difference'
else:
    wilcox_interp = 'Systematic difference detected'

# 3. Effect size
if abs(avg_cohens_d) < 0.20:
    effect_interp = 'Negligible practical difference'
elif abs(avg_cohens_d) < 0.50:
    effect_interp = 'Small practical difference'
elif abs(avg_cohens_d) < 0.80:
    effect_interp = 'Medium practical difference'
else:
    effect_interp = 'Large practical difference'

print(f'Pearson r: {avg_pearson:.3f}')
print(f'Spearman ρ: {avg_spearman:.3f}')
print(f'→ {corr_interp}')
print(f'\nWilcoxon p: {avg_wilcoxon_p:.4f}')
print(f'→ {wilcox_interp}')
print(f'\nCohen\'s d: {avg_cohens_d:.3f}')
print(f'→ {effect_interp}')

# Overall
if corr_interp == 'Formats largely equivalent' and wilcox_interp == 'No systematic difference' and abs(avg_cohens_d) < 0.20:
    final_interp = 'Formats largely equivalent'
else:
    final_interp = f'Formats partially overlapping: {corr_interp}, {effect_interp}'

print(f'\n✓ Final interpretation: {final_interp}')

---
## 5. TOKENOMICS & COST-EFFECTIVENESS (§ 3.5.4)

Evaluate cost-quality trade-offs using token counts and pricing.

### Analyses:
1. **Token & Cost Metrics:** Calculate tokens and costs per response
2. **Efficiency Metrics:** Cost per score point, cost per 1000 questions
3. **Visual Diagnostic:** Cost-quality bubble chart
4. **Interpretation:** Identify cost-effective models

### 5.1 Token Counting

In [ ]:
print('\n📊 Token Counting\n')

def count_tokens(text, encoding='cl100k_base'):
    """Count tokens using tiktoken"""
    if pd.isna(text) or text == '':
        return 0
    try:
        enc = tiktoken.get_encoding(encoding)
        return len(enc.encode(str(text)))
    except:
        return 0

# Add token counts
print('Counting tokens...')
osq_filtered['prompt_tokens'] = osq_filtered['osq_prompt'].apply(count_tokens)
osq_filtered['response_tokens'] = osq_filtered['model_response'].apply(count_tokens)
osq_filtered['total_tokens'] = osq_filtered['prompt_tokens'] + osq_filtered['response_tokens']

print('Token Statistics:')
print(osq_filtered[['prompt_tokens', 'response_tokens', 'total_tokens']].describe())

### 5.2 Cost Calculation (Model-Specific Pricing)

In [ ]:
print('\n📊 Cost Calculation\n')

# Model-specific pricing ($/1M tokens)
MODEL_PRICING = {
    'anthropic__claude-sonnet-4.5': {'input': 3.00, 'output': 15.00},
    'google__gemini-2.5-flash': {'input': 0.30, 'output': 2.50},
    'openai__gpt-4.1': {'input': 2.00, 'output': 8.00},
    'mistral-large__123b': {'input': 3.00, 'output': 9.00},
    'mistral-small3.2__24b': {'input': 0.06, 'output': 0.18},
    'llama3.2__1b': {'input': 0.02, 'output': 0.02},
    'llama3.2__3b': {'input': 0.018, 'output': 0.018},
    'llama3.3__70b': {'input': 0.10, 'output': 0.40},
    'llama4__16x17b': {'input': 0.22, 'output': 0.85},
    'mixtral__8x22b': {'input': 2.00, 'output': 6.00},
    'gemma3__1b': {'input': 0.01, 'output': 0.01},
    'gemma3__4b': {'input': 0.01703, 'output': 0.06815},
    'gemma3__12b': {'input': 0.03, 'output': 0.10},
    'gemma3__27b': {'input': 0.07, 'output': 0.50},
    'phi3.5__3.8b': {'input': 0.10, 'output': 0.10},
    'phi3__14b': {'input': 1.00, 'output': 1.00},
    'phi4-mini__3.8b': {'input': 0.05, 'output': 0.10},
    'phi4__14b': {'input': 0.07, 'output': 0.14},
    'devstral__24b': {'input': 0.10, 'output': 0.30},
}

def calculate_cost(row):
    """Calculate cost per response"""
    pricing = MODEL_PRICING.get(row['model'], {'input': 0, 'output': 0})
    input_cost = row['prompt_tokens'] * (pricing['input'] / 1_000_000)
    output_cost = row['response_tokens'] * (pricing['output'] / 1_000_000)
    return input_cost + output_cost

osq_filtered['cost_per_response'] = osq_filtered.apply(calculate_cost, axis=1)

print(f'Cost calculated for {len(osq_filtered)} responses')

### 5.3 Aggregate Tokenomics Metrics

In [ ]:
print('\n📊 Tokenomics Aggregation\n')

tokenomics = osq_filtered.groupby('model').agg({
    'prompt_tokens': 'mean',
    'response_tokens': 'mean',
    'total_tokens': 'mean',
    'cost_per_response': 'mean',
    'total_score': 'mean'
}).reset_index()

tokenomics['cost_per_1000'] = tokenomics['cost_per_response'] * 1000
tokenomics['cost_per_score'] = tokenomics['cost_per_response'] / (tokenomics['total_score'] / 100)

print('Tokenomics Summary (first 10):')
print(tokenomics.head(10).to_string(index=False))

# Export
tokenomics.to_csv(output_dir / 'table_tokenomics.csv', index=False)
export_latex_table(
    tokenomics[['model', 'response_tokens', 'cost_per_1000', 'total_score', 'cost_per_score']],
    'table_tokenomics.tex',
    'Tokenomics and Cost-Effectiveness Summary',
    'tab:tokenomics'
)

### 5.4 Visual: Cost-Quality Bubble Chart

In [ ]:
print('\n📊 PLOT 9: Cost-Quality Bubble Chart\n')

fig, ax = plt.subplots(figsize=(14, 10))

scatter = ax.scatter(
    tokenomics['cost_per_1000'],
    tokenomics['total_score'],
    s=tokenomics['response_tokens'] * 5,
    c=tokenomics['cost_per_score'],
    cmap='RdYlGn_r',
    alpha=0.6,
    edgecolors='black',
    linewidths=1.5
)

# Labels
for _, row in tokenomics.iterrows():
    ax.annotate(
        row['model'],
        (row['cost_per_1000'], row['total_score']),
        fontsize=8,
        ha='center'
    )

# Quadrants
median_cost = tokenomics['cost_per_1000'].median()
median_quality = tokenomics['total_score'].median()
ax.axvline(median_cost, color='gray', linestyle=':', alpha=0.5)
ax.axhline(median_quality, color='gray', linestyle=':', alpha=0.5)

# Quadrant labels
ax.text(ax.get_xlim()[0]*1.05, ax.get_ylim()[1]*0.98, 
        'High Quality\nLow Cost\n(Optimal)', 
        fontsize=10, ha='left', va='top', 
        color='green', fontweight='bold',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Cost per Score Point ($)', fontsize=12)

ax.set_xlabel('Cost per 1,000 Questions ($)', fontsize=12)
ax.set_ylabel('Mean OSQ Score (0-100)', fontsize=12)
ax.set_title('Cost-Quality Trade-off\n(Bubble size = avg response tokens)', 
             fontsize=14, fontweight='bold')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'fig_cost_quality_bubble.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'✅ Saved: {output_dir}/fig_cost_quality_bubble.png')

### 5.5 Tokenomics Interpretation

In [ ]:
print('\n📋 Tokenomics Interpretation\n')

# Best efficiency
best_efficiency = tokenomics.nsmallest(3, 'cost_per_score')[['model', 'cost_per_score', 'total_score']]
print('Most Cost-Efficient Models:')
print(best_efficiency.to_string(index=False))

# Best quality
best_quality = tokenomics.nlargest(3, 'total_score')[['model', 'total_score', 'cost_per_1000']]
print('\nHighest Quality Models:')
print(best_quality.to_string(index=False))

# Budget tiers
tokenomics['tier'] = pd.cut(
    tokenomics['cost_per_1000'],
    bins=[0, 10, 50, np.inf],
    labels=['Budget', 'Mid-tier', 'Premium']
)

print('\nModels by Cost Tier:')
print(tokenomics.groupby('tier')['model'].apply(list))

---
## 6. SUMMARY & EXPORT

Comprehensive summary of all analyses and export manifest.

### 6.1 Analysis Summary

In [ ]:
print('\n' + '='*80)
print('ANALYSIS SUMMARY')
print('='*80)

print('\n1. MCQ POSITION BIAS ANALYSIS')
print(f'   Models analyzed: {len(chi_square_df)}')
print(f'   Significant chi-square (p<0.05): {(chi_square_df["p_value"] < 0.05).sum()}')
print(f'   Mean Cramér\'s V: {chi_square_df["cramers_v"].mean():.4f}')
print(f'   Significant Friedman (p<0.05): {(friedman_df["p_value"] < 0.05).sum()}')

print('\n2. OSQ-SPECIFIC ANALYSIS')
print(f'   Models analyzed: {len(osq_stats_model)}')
print(f'   Judges: {len(COMPLETE_JUDGES)}')
print(f'   Mean OSQ score: {osq_filtered["total_score"].mean():.2f}')
if 'corr_df' in locals() and len(corr_df) > 0:
    print(f'   Avg inter-judge correlation: {corr_df["spearman_rho"].mean():.3f}')

print('\n3. FORMAT COMPARISON')
print(f'   Matched questions: {len(matched_question_ids)}')
print(f'   Pearson correlation (avg): {avg_pearson:.3f}')
print(f'   Spearman correlation (avg): {avg_spearman:.3f}')
print(f'   Cohen\'s d (avg): {avg_cohens_d:.3f}')

print('\n4. TOKENOMICS')
print(f'   Models analyzed: {len(tokenomics)}')
print(f'   Mean cost per 1000 questions: ${tokenomics["cost_per_1000"].mean():.2f}')
print(f'   Mean cost per score point: ${tokenomics["cost_per_score"].mean():.6f}')

print('\n' + '='*80)

### 6.2 Export Manifest

In [ ]:
import os

print('\n📁 EXPORT MANIFEST')
print('='*80)

# List all output files
output_files = sorted(output_dir.glob('*'))

print('\nFIGURES:')
figs = [f for f in output_files if f.suffix == '.png']
for i, f in enumerate(figs, 1):
    size_kb = os.path.getsize(f) / 1024
    print(f'  {i}. {f.name:50s} ({size_kb:8.1f} KB)')

print('\nTABLES (CSV):')
csvs = [f for f in output_files if f.suffix == '.csv']
for i, f in enumerate(csvs, 1):
    print(f'  {i}. {f.name}')

print('\nTABLES (LaTeX):')
texs = [f for f in output_files if f.suffix == '.tex']
for i, f in enumerate(texs, 1):
    print(f'  {i}. {f.name}')

print(f'\nTotal files: {len(output_files)}')
print('='*80)

### 6.3 Key Findings Summary

In [ ]:
findings = []

# Position bias
no_bias_models = interpretation_df[interpretation_df['category'] == 'No bias']['model'].tolist()
if len(no_bias_models) > 0:
    findings.append(f'Position Bias: {len(no_bias_models)} models show no position bias')
else:
    findings.append(f'Position Bias: All models show some degree of position bias')

# OSQ reliability
if 'avg_rho' in locals():
    if avg_rho >= 0.80:
        findings.append(f'OSQ Reliability: High (ρ={avg_rho:.3f})')
    elif avg_rho >= 0.50:
        findings.append(f'OSQ Reliability: Moderate (ρ={avg_rho:.3f})')
    else:
        findings.append(f'OSQ Reliability: Low (ρ={avg_rho:.3f})')

# Format comparison
findings.append(f'Format Alignment: {corr_interp} (r={avg_pearson:.3f})')

# Tokenomics
best_model = tokenomics.nsmallest(1, 'cost_per_score').iloc[0]
findings.append(f'Most Efficient: {best_model["model"]} (${best_model["cost_per_score"]:.6f}/point)')

print('\n📋 KEY FINDINGS')
print('='*80)
for i, finding in enumerate(findings, 1):
    print(f'{i}. {finding}')
print('='*80)

# Save to file
with open(output_dir / 'key_findings.txt', 'w') as f:
    f.write('KEY FINDINGS - Analysis v2\n')
    f.write('='*80 + '\n\n')
    for i, finding in enumerate(findings, 1):
        f.write(f'{i}. {finding}\n')

print(f'\n✅ Saved key findings to {output_dir}/key_findings.txt')

## ✅ ANALYSIS COMPLETE

All analyses from manuscript § 3.5 have been implemented:

1. ✅ MCQ Position Bias Analysis (§ 3.5.1)
2. ✅ OSQ-Specific Analysis (§ 3.5.2)
3. ✅ Format Comparison: MCQ vs OSQ (§ 3.5.3)
4. ✅ Tokenomics & Cost-Effectiveness (§ 3.5.4)

All figures, tables, and results have been exported to `output_v2/`.

**Next Steps:**
1. Review output files
2. Verify statistical results
3. Integrate tables and figures into manuscript
4. Commit changes to repository